## Astrophysics background

It is very common in Astrophysics to work with sky pixels. The sky is tassellated in patches with specific properties and a sky map is then a collection of intensity values for each pixel. The most common pixelization used in Cosmology is [HEALPix](http://healpix.jpl.nasa.gov).

Measurements from telescopes are then represented as an array of pixels that encode the pointing of the instrument at each timestamp and the measurement output.

## Sample timeline

For simplicity let's assume we have a sky with 50K pixels:

In [ ]:
import pandas as pd
import numba
import numpy as np

NPIX = 50000
NTIME = int(50 * 1e6)

# The pointing of our instrument is an array of pixels, random in our sample case:
pixels = np.random.randint(0, NPIX - 1, NTIME)
# Our data are also random:
timeline = np.random.randn(NTIME)

## Create a map of the sky with pandas

One of the most common operations is to sum all of our measurements in a sky map, so the value of each pixel in our sky map will be the sum of each individual measurement.
The easiest way is to use the `groupby` operation in `pandas`:

In [ ]:
timeline_pandas = pd.Series(timeline, index=pixels)
%time m = timeline_pandas.groupby(level=0).sum()

## Your turn — write the groupby kernel

Write a **pure-Python** function `groupby_python(index, value, output)` that fills `output` so that `output[p]` is the sum of all `value[i]` where `index[i] == p`.

This is the same operation as the pandas `groupby(...).sum()` above — we are reimplementing it so we can hand it to Numba.

```python
def groupby_python(index, value, output):
    for i in range(index.shape[0]):
        # your code: accumulate value[i] into output[...]
        pass
```

Try it yourself, then check the solution below.

In [ ]:
# Solution
def groupby_python(index, value, output):
    for i in range(index.shape[0]):
        output[index[i]] += value[i]

m_python = np.zeros_like(m)
%time groupby_python(pixels, timeline, m_python)
np.testing.assert_allclose(m_python, m)  # same answer as pandas

Pure Python is slower than the `pandas` version implemented in `cython`.

Now compile it with `numba.jit`:

In [ ]:
groupby_numba = numba.jit(groupby_python, nopython=True)

m_numba = np.zeros_like(m)
groupby_numba(pixels, timeline, m_numba)  # first call compiles

m_numba = np.zeros_like(m)
%time groupby_numba(pixels, timeline, m_numba)
np.testing.assert_allclose(m_numba, m)

Performance improvement is about 50x compared to Python and up to 10x compared to Pandas, pretty good!

## Use `numba.jit` as a decorator

The exact same result is obtained if we use `numba.jit` as a decorator:

In [ ]:
@numba.jit(nopython=True)
def groupby_numba(index, value, output):
    for i in range(index.shape[0]):
        output[index[i]] += value[i]

## Example of `numba` in a Python library

* https://github.com/galsci/pysm/blob/2b69973bc8d6bd7a9d5c861477d015461fde185a/pysm3/models/power_law.py#L101-L132
* https://github.com/galsci/pysm/blob/2b69973bc8d6bd7a9d5c861477d015461fde185a/pysm3/models/dust.py#L109-L157